# Docker Fundamentals

---

In this notebook, we will learn the core concepts and commands for **Docker**, the tool that lets you package applications into portable, reproducible containers.

We will cover:
- Installing Docker
- Understanding images, containers, and layers
- Writing a Dockerfile from scratch
- Building and running containers
- Essential Docker CLI commands
- Layer caching and building optimization
- Environment variables and port mapping

> ⚠️ **Note:** Docker commands are run in the **terminal**, not in Jupyter Notebook cells. This notebook explains the concepts and shows the commands. You should type and run them in your terminal.

---

## 1. Installation

Install **Docker Desktop** from [docker.com/get-started](https://www.docker.com/get-started/). It's available for Windows, macOS, and Linux.

After installation, verify it's working.

In [ ]:
docker --version
# Docker version 29.x.x, build ...

docker run hello-world
# Should print: Hello from Docker!

The `hello-world` command downloads a tiny test image and runs it. If you see the greeting, Docker is set up correctly.

---

## 2. The Dockerfile: Line by Line

A `Dockerfile` is a plain text file (no extension) that tells Docker how to build an image. Here is a minimal example for a Python application.

```dockerfile
# 1. Start from a base image
FROM python:3.13-slim

# 2. Set the working directory inside the container
WORKDIR /app

# 3. Copy the requirements file and install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 4. Copy the rest of the application code
COPY . .

# 5. Expose the port the app will listen on
EXPOSE 8000

# 6. Define the command to run when the container starts
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Instruction Reference

| **Instruction** | **Purpose** | **Notes** |
| :--- | :--- | :--- |
| `FROM` | Sets the **base image**, the starting point for your image. | `python:3.13-slim` is a lightweight Debian image with Python pre-installed. Always use `-slim` or `-alpine` to keep images small |
| `WORKDIR` | Sets the current directory inside the container | All subsequent `COPY`, `RUN`, `CMD` commands run relative to this. Similar to `cd` |
| `COPY` | Copies files from your machine into the image | `COPY requirements.txt .` copies the file into `/app/` (the `WORKDIR`) |
| `RUN` | Executes a command during the **build** phase. | Used to install packages. `--no-cache-dir` avoids caching pip packages inside the image (saves space) |
| `EXPOSE` | Documents whic port the app uses | This is informational. You still need to publish the port with `-p` when running |
| `CMD` | Defines the **default command** when the container starts | Only one `CMD` per Dockerfile. Use JSON array format ("exec form") for proper signal handling. |

---

## 3. Building an Image

The `docker build` command reads a Dockerfile and creates an image.

In [ ]:
docker build -t my-app:1.0 .

| **Part** | **Meaning** |
| :--- | :--- |
| `docker build` | Build an image from a Dockerfile |
| `-t my-app:1.0` | **Tag** the image with a name (`my-app`) and version (`1.0`). Without a tag, it gets a random hash. | 
| `.` | The **build context**, the directory Docker uses as the root for `COPY` commands. Usually the current directory. |

### What Happens During Build
Docker processes the Dockerfile **top to bottom**, creating a read-only **layer** for each instruction:

```
Layer 1: FROM python:3.13-slim          (base OS + Python)
Layer 2: WORKDIR /app                   (set working directory)
Layer 3: COPY requirements.txt .        (add requirements file)
Layer 4: RUN pip install ...            (install dependencies)
Layer 5: COPY . .                       (add application code)
```

**Layer caching** is why the order of your Dockerfile matters.

---

## 4. Layer Caching (Why Order Matters)

Docker caches each layer. If a layer hasn't changed since the last build, Docker reuses the cached version instead of rebuilding it.

**The golden rule:** Put things that change **least frequently** at the top, and things that change **most frequently** at the bottom.

### ❌ Bad Order

```dockerfile
COPY . .                                # Copy ALL files (including source code)
RUN pip install -r requirements.txt     # Install dependencies
```

Every time you change **any** source file, the `COPY` layer is invalidated, which also invalidates the `pip install` layer. Dependencies are reinstalled from scratch every time.

### ✅ Good Order

```dockerfile
COPY requirements.txt .                 # Copy ONLY requirements first
RUN pip install -r requirements.txt     # Install dependencies (cached if requirements unchanged)
COPY . .                                # Copy source code
```

Now when you change source code, only the last `COPY` layer is rebuilt. The `pip install` layer stays cached because `requirements.txt` didn't change. This saves minutes on every build.

---

## 5. Running a container

Once you've built an image, run it:

In [ ]:
docker run -p 8000:8000 my-app:1.0

| **Part** | **Meaning** |
| :--- | :--- |
| `docker run` | Create and start a new container from an image |
| `-p 8000:8000` | **Port mapping:** map port 8000 on your machine (left) to port 8000 inside the container (right). Without this, the container's port is inaccessible. |
| `my-app:1.0` | The image to run |

### Common Run Flags

| **Flag** | **Purpose** | **Example** |
| :--- | :--- | :--- |
| `-d` | Run in **detached** mode (background) | `docker run -d -p 8000:8000 my-app` |
| `--name` | Give the container a friendly name | `docker run --name iris-api my-app` |
| `-e` | Set **environment variables** | `docker run -e MODEL_PATH=/app/model.joblib my-app` |
| `--rm` | Automatically **remove** the container when it stops | `docker run --rm my-app` |
| `-v` | Mount a **volume** (share files between host and container) | `docker run -v ./data:/app/data my-app` |

---

## 6. Essential CLI Commands

### Image Commands

| **Command** | **What It Does** |
| :--- | :--- |
| `docker images` | List all images on your machine |
| `docker build -t name:tag .` | Build an image from a Dockerfile |
| `docker rmi image_name` | Remove an image | 
| `docker image prune` | Remove all unused (dangling) images |

### Container Commands

| **Command** | **What It Does** |
| :--- | :--- |
| `docker ps` | List **running** containers |
| `docker ps -a` | List **all** containers (including stopped) |
| `docker stop container_name` | Stop a running container |
| `docker rm container_name` | Remove a stopped container |
| `docker logs container_name` | View the container's stdout/stderr output | 
| `docker exec -it container_name bash` | Open an interactive shell inside a running container |

### Cleanup

Docker images and containers accumulate over time. To clean up everythin:

In [ ]:
# Remove all stopped containers, unused networks, dangling images, and build cache
docker system prune

# Nuclear option: remove EVERYTHING (including named images)
docker system prune -a


---

## 7 The .dockerignore File

Just like `.gitignore` prevents files from being tracked by Git, `.dockerignore` prevents files from being copied into the Docker build context. This makes builds faster and images smaller

```
__pycache__
*.pyc
.git
.venv
*.ipynb
.ipynb_checkpoints
```

Without a `.dockerignore`, `COPY . .` would copy your entire directory into the image, including `.git/`, virtual environments, and Jupyter checkpoints.

---

## 8. Docker Compose

Docker Compose is a tool for defining multi-container applications (or just simplifying single-container setups) in a YAML file.

Instead of typing:

In [ ]:
docker build -t iris-api:1.0 .
docker run -d -p 8000:8000 --name iris-api iris-api:1.0

You write a `docker-compose.yml`

```YAML
services:
  api:
    build: .
    ports:
      - "8000:8000"
```

Then run:

In [ ]:
docker compose up --build

### Compose Commands

| **Command** | **What It Does** |
| :--- | :--- |
| `docker compose up` | Start all services |
| `docker compose up --build` | Rebuild images and start |
| `docker compose up -d` | Start in detached mode (background) |
| `docker compose down` | Stop and remove all containers |
| `docker compose logs` | View logs from all services | 

---

## 9. Summary

| **Concept** | **Key Takeaway** |
| :--- | :--- |
| **Image** | A read-only blueprint from a Dockerfile |
| **Container** | A running instance of an image. |
| **Dockerfile** | Step-by-step instructions to build an image (`FROM`, `COPY`, `RUN`, `CMD`). |
| **Layer Caching** | Put rarely-changing instructions (dependencies) **before** frequently-changing ones (source code). |
| `-p host:container` | Maps a port on your machine to a port inside the container |
| `.dockerignore` | Exclude unnecessary files from the build context |
| **Docker Compose** | Define and run containers with a YAML file instead of long CLI commands. |

---

**Next:** [Dockerizing an ML API](./02_dockerizing_an_ml_api.ipynb) — Applying everything we learned to containerize our Iris FastAPI application.